In [1]:
# Milestone 2: Email Assistant Evaluation
# Name: Arbind
# ===== SIMPLE DATASET LOADING =====
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
print("Dataset shape:", df.shape)
print("\nColumns confirmed:")
print("-" * 30)
print("✓ body")
print("✓ ideal_intent") 
print("✓ ideal_tone")

print("\nFirst 5 rows:")
df[['id', 'subject','body', 'ideal_intent', 'ideal_tone' ]].head()



Dataset shape: (200, 8)

Columns confirmed:
------------------------------
✓ body
✓ ideal_intent
✓ ideal_tone

First 5 rows:


,id,subject,body,ideal_intent,ideal_tone
0,1,Password Reset Request,Reminder: The client meeting is scheduled at 1...,respond,neutral
1,2,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,respond,neutral
2,3,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,respond,neutral
3,4,Monthly Report,"Hello team, please find the attached weekly re...",respond,polite
4,5,Survey,"Hello team, please find the attached weekly re...",respond,polite


In [2]:
# PART 3: Email Assistant Logic (from Milestone 1)
def email_assistant(email_text):
    """
    Email classification logic from Milestone 1
    Returns: (intent, tone)
    """
    text = str(email_text).lower()
    
    # Security/urgent cases
    if any(word in text for word in ["login", "password", "security", "suspended", "verification"]):
        return "notify", "urgent"
    elif "urgent" in text or "deadline" in text or "submit" in text:
        return "notify", "urgent"
    
    # Thank you / ignore cases
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    
    # Default case
    else:
        return "respond", "neutral"

print("✅ Email assistant function loaded")


✅ Email assistant function loaded


In [3]:
# PART 4: Generate Predictions
print("Generating predictions for 200 emails...")
predictions = []

for idx, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
print("Predictions generated:")
print(pred_df.head())
print(f"Prediction shape: {pred_df.shape}")



Generating predictions for 200 emails...
Predictions generated:
   id predicted_intent predicted_tone
0   1          respond        neutral
1   2          respond        neutral
2   3          respond        neutral
3   4          respond        neutral
4   5          respond        neutral
Prediction shape: (200, 3)


In [4]:
# PART 5: Evaluate Accuracy
print("Merging predictions with ground truth...")
eval_df = df.merge(pred_df, on="id")
print("Merged dataset shape:", eval_df.shape)

def evaluate(row):
    """Calculate score: 1 point per correct prediction (intent + tone = 2 max)"""
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

eval_df["score"] = eval_df.apply(evaluate, axis=1)

# Calculate accuracy
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
print(f"\n📊 FINAL ACCURACY: {accuracy:.2f}%")
print(f"Total correct predictions: {eval_df['score'].sum()}/{len(eval_df) * 2}")


Merging predictions with ground truth...
Merged dataset shape: (200, 10)

📊 FINAL ACCURACY: 78.75%
Total correct predictions: 315/400


In [5]:
# PART 6: Save Evaluation Output
output_filename = "../data/milestone2_output_arbind2.csv"
eval_df.to_csv(output_filename, index=False)

print(f"✅ Evaluation saved: {output_filename}")
print("\nSample evaluation results:")
eval_df[['id', 'subject', 'ideal_intent', 'ideal_tone', 
               'predicted_intent', 'predicted_tone', 'score']].head(10)


✅ Evaluation saved: ../data/milestone2_output_arbind2.csv

Sample evaluation results:


,id,subject,ideal_intent,ideal_tone,predicted_intent,predicted_tone,score
0,1,Password Reset Request,respond,neutral,respond,neutral,2
1,2,Congratulations! You've Won,respond,neutral,respond,neutral,2
2,3,Promotion: Big Sale,respond,neutral,respond,neutral,2
3,4,Monthly Report,respond,polite,respond,neutral,1
4,5,Survey,respond,polite,respond,neutral,1
5,6,Unusual Login Attempt,respond,neutral,respond,neutral,2
6,7,Project Update,ignore,neutral,respond,neutral,1
7,8,Project Update,respond,polite,respond,neutral,1
8,9,Account Suspended,notify,urgent,notify,urgent,2
9,10,Monthly Report,respond,neutral,respond,neutral,2


In [6]:
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,[alerts@bank.com](mailto:alerts@bank.com),Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,[alerts@bank.com](mailto:alerts@bank.com),Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral
2,3,[no-reply@service.com](mailto:no-reply@service...,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,[sales@shop.com](mailto:sales@shop.com),Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,[no-reply@service.com](mailto:no-reply@service...,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite
